In [ ]:
!rm -rf /kaggle/working/Real-ESRGAN
!git clone --depth 1 --branch master https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile inference.py inference/*.py encode/*.py inference/models/*.py
!cd /kaggle/working/Real-ESRGAN && python inference.py --help >/dev/null
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E "(libx265|hevc_nvenc|libsvtav1|libaom-av1|av1_nvenc)" || true


In [ ]:
import torch

INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime/ts_2.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2

# RIFE_FPS = 0 关闭 RIFE 并保持源帧率；>0 时必须 >= 输入视频源帧率。
RIFE_FPS = 60

START_TIME = 5 * 60 + 35
TEST_SECONDS = 15

# GPU：False=单 GPU(cuda:0)，True=双 GPU(cuda:0,1)。
DUAL_GPU = False
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 1:
    raise RuntimeError("No CUDA GPU is visible to PyTorch")
if DUAL_GPU and GPU_COUNT < 2:
    raise RuntimeError(
        f"DUAL_GPU=True requires at least 2 visible CUDA GPUs; got {GPU_COUNT}"
    )
GPU_IDS = "0,1" if DUAL_GPU else "0"

# BasicVSR++ 固定参数。
BVS_TILE_SIZE = 640
BVS_CLIP_LENGTH = 13
BVS_BATCH_SIZE = 1
BVS_STRENGTH = 1.0

# 流水线：BasicVSR++ -> RIFE 4.25 -> Real-ESRGAN -> Lanczos4 -> encoder。

# 编码器：
#   CPU HEVC = "libx265"
#   GPU HEVC = "hevc_nvenc"
#   CPU AV1  = "libsvtav1"
#   CPU AV1  = "libaom-av1"
#   GPU AV1  = "av1_nvenc"
VIDEO_CODEC = "hevc_nvenc"
CRF = 18
PRESET = "medium"
SVTAV1_PRESET = 6
AOM_CPU_USED = 6
CQ = 18
NVENC_PRESET = "p7"
ENCODE_GPU = 0

AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"


In [ ]:
import subprocess
import sys

command = [
    sys.executable, "/kaggle/working/Real-ESRGAN/inference.py",
    "--input", INPUT_VIDEO,
    "--output", OUTPUT_VIDEO,
    "--model", MODEL,
    "--model-path", MODEL_PATH,
    "--scale", str(SCALE),
    "--rife-fps", str(RIFE_FPS),
    "--gpu-ids", GPU_IDS,
    "--bvs-tile-size", str(BVS_TILE_SIZE),
    "--bvs-clip-length", str(BVS_CLIP_LENGTH),
    "--bvs-batch-size", str(BVS_BATCH_SIZE),
    "--bvs-strength", str(BVS_STRENGTH),
    "--video-codec", VIDEO_CODEC,
    "--crf", str(CRF),
    "--preset", PRESET,
    "--svtav1-preset", str(SVTAV1_PRESET),
    "--aom-cpu-used", str(AOM_CPU_USED),
    "--cq", str(CQ),
    "--nvenc-preset", NVENC_PRESET,
    "--encode-gpu", str(ENCODE_GPU),
    "--audio-codec", AUDIO_CODEC,
    "--audio-bitrate", AUDIO_BITRATE,
    "--start-time", str(START_TIME),
    "--test-seconds", str(TEST_SECONDS),
    "--ffmpeg-bin", "ffmpeg",
    "--ffprobe-bin", "ffprobe",
]

# Capture the child process once and relay a single merged stream through
# the notebook kernel. This avoids duplicate persisted Kaggle log records.
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
returncode = process.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)
